# LAB 9: Full RAG solution with LlamaIndex and Milvus
In this lab we are going to build a full RAG solution using the LlamaIndex framework that leverages models from the Nvidia NIM API catalog (including LLM and embedding models). For this, you will need an API key from NVIDIA

<font color="red"><b>IMPORTANT</b></font>: This notebook requires Python 3.11 or older. This can be done by changing the "runtime". Go to "Runtime" > "Change runtime type" and select "2025-07". This will use a kernel with Python 3.11.13

In [ ]:
!python3 --version

## Install dependencies

The first step is to install the necessary libraries. This installs the core llama-index package which draws a lot of dependencies.

In [ ]:
!pip install llama-index

If you look carefully at the previous output you will notice that the only LLM interface that has been installed is OpenAI. We are going to use NVIDIA NIMs including LLMs and Embedding models, so we need to install their corresponding modules.

You can see what LLM modules are available in LlamaIndex in [https://docs.llamaindex.ai/en/stable/module_guides/models/llms/modules/](https://docs.llamaindex.ai/en/stable/module_guides/models/llms/modules/)

In [ ]:
!pip install llama-index-llms-nvidia llama-index-embeddings-nvidia

Finally, we need to install Milvus itself and the LlamaIndex library to interact with Milvus. For simplicity, in this lab we will use "Milvus Lite". Consider using the standalone or cluster versions for production use cases.

In [ ]:
pip install llama-index-vector-stores-milvus pymilvus[milvus-lite]

Now we can import the components we need for this lab.

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext
from llama_index.core import Settings
from llama_index.llms.nvidia import NVIDIA
from llama_index.embeddings.nvidia import NVIDIAEmbedding
from llama_index.vector_stores.milvus import MilvusVectorStore
import os
import urllib3
urllib3.disable_warnings()

## Read environment variables

A best practice for managing information like the API key is to provide it as an environment variable to the application and get the application to read it from the environment. This is typically done as follows:
```
apikey = os.environ["NVIDIA_API_KEY"]
```


But inside Google Colab we can add them as secrets clicking on the little "key" symbol on the left menu and then read them with "userdata".

In [ ]:
from google.colab import userdata
apikey = userdata.get('apikey')

## Instantiate the LLM

LlamaIndex provides a "Settings" object that stores the most commonly used resources in a LlamaIndex workflow, ex: llm, embed_model. It is a sort of a global storage place for all default settings. If one attribute is not provided anywhere else in the code, the "Settings" object will be queried. As you will see there are several default values that are assumed if not specified. This makes the code look cleaner.

The following line will be sufficient to instantiate an LLM from the Nvidia NIM API if the right defaults apply.

In [ ]:
#Settings.llm = NVIDIA(base_url = llmurl, api_key = apikey)
Settings.llm = NVIDIA(api_key = apikey)

You must define the parameter ```base_url``` when working with NIM's running locally in your environment. If this parameter is not explicitly defined then ```NVIDIA()``` assumes ```base_url = "https://integrate.api.nvidia.com/v1"```

If the parameter ```api_key``` is not omitted then it will try to read the variable ```NVIDIA_API_KEY``` from the environment

Also, like in previous NIM examples, if ```model``` is not present then it will assume ```meta/llama3-8b-instruct```

So for example, let's say you want to connect to a NIM that is running locally in your datacenter, you want to hard-code the key explicitly in your code and you want to use a Mistral model. Then, the Settings would look like this
```
Settings.llm = NVIDIA(
    base_url="http://nim-host-address:8000/v1",
    api_key = "nvapi-123456789abcdefg",
    model="mistralai/mistral-7b-instruct-v0.2")
```


We can verify what model we are pointing to

In [ ]:
print("... Using: ", Settings.llm.model)

## Instantiate the embedding model

We are going to use the "Settings" object again but this time for the embedding model. Instead of requesting a specific embedding model, we leave ```model``` blank and it will select the default one for the Nvidia module, typically 'nv-embedqa-e5-v5'.

Also, similar comments apply here to the need to of defining base_url when using embedding model NIM locally

In [ ]:
#Settings.embed_model = NVIDIAEmbedding(base_url = embedurl, api_key = apikey, truncate="END")
Settings.embed_model = NVIDIAEmbedding(api_key = apikey, truncate="END")
print("... Using: ", Settings.embed_model.model)

## Milvus section

We are going to instantiate a Milvus vector store. The dimensions need to match the number of dimensions of the embeddings generated by the embedding model we have just chosen above. A quick Google search reveals Nvidia's "nv-embedqa-e5-v5" creates embeddings with 1024 dimensions.

If omitted, the "overwrite" parameter is set to "False". Change it to "True" if you are testing and want to replace the existing collection.

Milvus offers lots of functionality so there many more parameters available. You can check them in the <a href="https://developers.llamaindex.ai/python/framework-api-reference/storage/vector_store/milvus/"  target="_blank">LlamaIndex documentation</a>.

In [ ]:
vector_store = MilvusVectorStore(
    uri="./milvus_demo.db",
    dim=1024,
    collection_name="my_collection",
    overwrite=False
    )

storage_context = StorageContext.from_defaults(vector_store=vector_store)

## Load the documents

Let's use "Simple Directory Reader" to load the documents in the "data" directory. The data directory could also be a mounted directory pointing to PowerScale.

SimpleDirectoryReader will ingest Markdown, PDFs, Word documents, PowerPoint decks, images, audio and video from the specified directory

In [ ]:
# Ensure the "data" directory exist before running the following command
wget 'https://raw.githubusercontent.com/run-llama/llama_index/main/docs/docs/examples/data/paul_graham/paul_graham_essay.txt' -O 'data/paul_graham_essay.txt'
documents = SimpleDirectoryReader("data").load_data()

## Create an Index

Index is a key contruct in the LlamaIndex framework. To build it we need "Documents", an "Embedding model" and a "Storage Context". "Storage Context" could be omited if we want to use the simple in-memory vector store LlamaIndex provides for quick experimentation. But in this case we are going to use Milvus, so we need "storage_context"

Notice also how we are not specifying the ```embed_model``` parameter because it is already defined in the ```Settings``` object

In [ ]:
index = VectorStoreIndex.from_documents(documents, storage_context=storage_context)

## Build the query engine

The final step is to use the "LLM" and the "Index" to build the "Query Engine". Thanks again to the ```Setttings``` object, we don't need to specify ```llm=llm``` or ```embed_model=embed_model```

In [ ]:
query_engine = index.as_query_engine()

## Query the RAG solution

Everything is ready to start querying our RAG solution. We use the ```.query()``` method from the ```query_engine```

In [ ]:
response = query_engine.query("What did the author do growing up?")
print(response)

## End of Lab 9